In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import IntSlider, VBox, interactive_output

# Configuration
N_max = 64
f1, f2 = 0.1, 0.28 

def plot_full_analysis(N):
    plt.close('all')
    
    # Time domain
    stretched_n = np.linspace(0, N_max - 1, N)
    x_finite = 3.0 * np.sin(2 * np.pi * f1 * np.arange(N)) + 1.5 * np.cos(2 * np.pi * f2 * np.arange(N))
    
    # FFT settings
    n_fft = 2048 
    freqs = np.fft.fftshift(np.fft.fftfreq(n_fft))
    
    # Spectra
    x_padded = np.zeros(n_fft)
    x_padded[:N] = x_finite
    X_spec = np.fft.fftshift(np.fft.fft(x_padded, n_fft))
    
    ideal_mag = np.zeros(n_fft)
    impulse_indices = [
        np.argmin(np.abs(freqs - f1)), np.argmin(np.abs(freqs + f1)),
        np.argmin(np.abs(freqs - f2)), np.argmin(np.abs(freqs + f2))
    ]
    for idx in impulse_indices:
        ideal_mag[idx] = N * 1.5 
    
    w_padded = np.zeros(n_fft)
    w_padded[:N] = np.ones(N)
    W_spec = np.fft.fftshift(np.fft.fft(w_padded, n_fft))
    
    # Plotting (Height reduced by 1/3: 14 * 2/3 ≈ 9.33)
    fig = plt.figure(figsize=(16, 9.33))
    gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 1], hspace=0.4, wspace=0.2)
    
    # Time Domain
    ax_t = fig.add_subplot(gs[0, :])
    ax_t.stem(stretched_n, x_finite, linefmt='#1f77b4', markerfmt='o', basefmt='k-')
    ax_t.set_title(f"Finite Signal in Time Domain (N={N})", fontsize=10, fontweight='bold')
    ax_t.set_xlim(-1, N_max)
    ax_t.grid(True, linestyle='--')
    
    # Window Magnitude & Phase
    ax_wm = fig.add_subplot(gs[1, 0])
    ax_wm.plot(freqs, np.abs(W_spec), 'orange')
    ax_wm.set_title("Rectangular Window: Magnitude", fontsize=9)
    ax_wm.grid(True, linestyle='--')
    
    ax_wp = fig.add_subplot(gs[2, 0])
    ax_wp.plot(freqs, np.angle(W_spec), 'purple')
    ax_wp.set_title("Rectangular Window: Phase", fontsize=9)
    ax_wp.grid(True, linestyle='--')
    
    # Signal Magnitude Comparison
    ax_sm = fig.add_subplot(gs[1, 1])
    ax_sm.plot(freqs, np.abs(X_spec), 'r-', label='Finite (Windowed)', lw=1.2)
    ax_sm.stem(freqs, ideal_mag, linefmt='k--', markerfmt=' ', basefmt=' ', label='Ideal (Infinite)')
    ax_sm.set_title("Magnitude Spectrum (Spectral Leakage)", fontsize=9, fontweight='bold')
    ax_sm.set_xlim(-0.5, 0.5)
    ax_sm.legend(fontsize=8)
    ax_sm.grid(True, linestyle='--')
    
    # Signal Phase
    ax_sp = fig.add_subplot(gs[2, 1])
    ax_sp.plot(freqs, np.angle(X_spec), 'g-', label='Finite (Windowed)')
    ax_sp.vlines(x=[f1, -f1, f2, -f2], ymin=-3, ymax=3, colors='k', linestyles='--')
    ax_sp.set_title("Phase Spectrum", fontsize=9, fontweight='bold')
    ax_sp.set_xlim(-0.5, 0.5)
    ax_sp.grid(True, linestyle='--')

    plt.show()

# Widgets
N_slider = IntSlider(value=32, min=8, max=64, step=4, description='Window N:')
display(VBox([N_slider, interactive_output(plot_full_analysis, {'N': N_slider})]))